# ErgoVision — Ergonomic Risk Assessment

Vision-based ergonomic risk assessment using YOLOv8-pose + RULA-inspired scoring.

**Esecuzione su Kaggle**: monta il tuo dataset (immagini o video) nell'input di Kaggle
e il notebook lo trova automaticamente in `/kaggle/input/`.

## 1. Installazione dipendenze

In [ ]:
!pip install -q ultralytics opencv-python-headless numpy matplotlib

## 2. Import e configurazione

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '.')

import ergovision
from ergovision import ErgoPipeline
from ergovision.data.video_frame_extractor import VideoFrameExtractor
from ergovision.config import IMAGE_EXTENSIONS, VIDEO_EXTENSIONS

print(f'ErgoVision v{ergovision.__version__} loaded')

## 3. Rilevamento dataset

Cerca automaticamente immagini o video in `/kaggle/input/`.

In [ ]:
input_root = Path('/kaggle/input')

candidates = sorted(input_root.iterdir()) if input_root.exists() else []
if candidates:
    DATA_PATH = candidates[0]
    print(f'Dataset rilevato: {DATA_PATH}')
else:
    raise FileNotFoundError(
        'Nessun dataset trovato in /kaggle/input/. '
        'Aggiungi il dataset al notebook.'
    )

n_video = sum(1 for _ in DATA_PATH.rglob('*') if _.suffix in VIDEO_EXTENSIONS)
n_img = sum(1 for _ in DATA_PATH.rglob('*') if _.suffix in IMAGE_EXTENSIONS)
print(f'  Video:    {n_video}')
print(f'  Immagini: {n_img}')

## 4. Esecuzione pipeline

Se ci sono **video**: estrae i frame, poi esegue la pipeline.
Se ci sono **immagini**: esegue la pipeline direttamente.

In [ ]:
OUTPUT_DIR = Path('outputs/kaggle')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pipeline_input = DATA_PATH

if n_video > 0 and n_img == 0:
    print('MODALITA VIDEO: estrazione frame...')
    frames_dir = OUTPUT_DIR / 'frames'
    extractor = VideoFrameExtractor(
        output_dir=frames_dir,
        sampling_rate=1.0,
        max_frames_per_video=100,
        blur_threshold=0,
    )
    video_files = []
    for ext in VIDEO_EXTENSIONS:
        video_files.extend(DATA_PATH.rglob(f'*{ext}'))
    for v in video_files:
        result = extractor.extract(v)
        print(f'  {v.name}: {result.frames_extracted} frame')
    pipeline_input = frames_dir

elif n_img > 0:
    print('MODALITA IMMAGINI: uso diretto...')
else:
    print('Nessun file da elaborare.')

print(f'\nPipeline input: {pipeline_input}')

In [ ]:
import shutil

pipeline = ErgoPipeline()
results = pipeline.run(
    dataset_path=pipeline_input,
    subset_size=9999,
    save_visualizations=True,
    verbose=True,
)

# Copia risultati in outputs/kaggle/
src_csv = Path('outputs') / 'ergonomic_assessment.csv'
src_json = Path('outputs') / 'ergonomic_assessment.json'
src_viz = Path('outputs') / 'visualizations'

if src_csv.exists():
    shutil.copy2(src_csv, OUTPUT_DIR / 'ergonomic_assessment.csv')
if src_json.exists():
    shutil.copy2(src_json, OUTPUT_DIR / 'ergonomic_assessment.json')
if src_viz.exists():
    dest_viz = OUTPUT_DIR / 'visualizations'
    if dest_viz.exists():
        shutil.rmtree(dest_viz)
    shutil.copytree(src_viz, dest_viz)

# Pulisce output temporanei
for f in [src_csv, src_json]:
    if f.exists():
        f.unlink()
if src_viz.exists():
    shutil.rmtree(src_viz)

print(f'\nRisultati salvati in: {OUTPUT_DIR.resolve()}')

## 5. Preview risultati

In [ ]:
import csv
import matplotlib.pyplot as plt
import cv2
from collections import Counter

csv_path = OUTPUT_DIR / 'ergonomic_assessment.csv'
if csv_path.exists():
    with open(csv_path, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    print(f'Righe totali: {len(rows)}')

    if rows:
        cols = ['file', 'classe', 'score', 'torso', 'neck', 'knee']
        print(f'{cols[0]:30s} {cols[1]:13s} {cols[2]:5s} {cols[3]:6s} {cols[4]:6s} {cols[5]:6s}')
        print('-' * 66)
        for r in rows[:8]:
            name = Path(r['image_path']).name[:29]
            print(f'{name:30s} {r["final_risk_class"]:13s} {r["final_score"]:5s} '
                  f'{r["torso_angle"]:>6s} {r["neck_angle"]:>6s} {r["knee_angle"]:>6s}')
        if len(rows) > 8:
            print(f'... e {len(rows) - 8} righe in piu')

    # Distribuzione
    classes = [r['final_risk_class'] for r in rows]
    counts = Counter(classes)

    fig, ax = plt.subplots(figsize=(6, 4))
    colors = ['#2ecc71', '#f1c40f', '#e74c3c']
    bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='gray')
    ax.set_ylabel('Rilevazioni')
    ax.set_title('Distribuzione Rischio Ergonomico')
    for bar, cnt in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(cnt), ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Visualizzazioni
    viz_dir = OUTPUT_DIR / 'visualizations'
    viz_files = sorted(viz_dir.glob('*'))[:6]
    if viz_files:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        for ax, vf in zip(axes.flat, viz_files):
            img = cv2.cvtColor(cv2.imread(str(vf)), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(vf.name, fontsize=9)
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('Nessun risultato. Esegui la pipeline prima.')

---
Tutti gli output sono in `outputs/kaggle/`